In [ ]:
!pip install Pillow



In [ ]:
import os
import cv2
from ultralytics import YOLO
from tqdm import tqdm
from PIL import Image # Импортируем Pillow

# 1. Пути к файлам (измени на свои)
model_path = '/content/runs/detect/train3/weights/best.pt'
input_folder = '/content/wine_images1' # Папка с webp файлами
output_folder = '/content/new_labels' # Куда сохранять (будут JPG)

# 2. Загрузка модели
model = YOLO(model_path)
os.makedirs(output_folder, exist_ok=True)

# 3. Список файлов (берем всё, что похоже на картинки)
images = [f for f in os.listdir(input_folder) if f.endswith(('.webp', '.jpg', '.png', '.jpeg'))]

print(f"Найдено изображений: {len(images)}. Начинаем обрезку...")

# 4. Проход по картинкам с конвертацией
for img_name in tqdm(images):
    img_path = os.path.join(input_folder, img_name)

    try:
        # ОТКРЫВАЕМ ЧЕРЕЗ PILLOW И КОНВЕРТИРУЕМ В JPG В ПАМЯТИ
        pil_image = Image.open(img_path).convert('RGB')

        # КОНВЕРТИРУЕМ В МАССИВ ДЛЯ OPENCV (BGR)
        import numpy as np
        img = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)

        # 5. Запуск предсказания
        results = model(img, verbose=False)

        # Обработка результатов (как было раньше)
        for r in results:
            boxes = r.boxes
            if len(boxes) > 0:
                box = boxes[0].xyxy[0].tolist()
                x1, y1, x2, y2 = map(int, box)
                cropped_img = img[y1:y2, x1:x2]

                # 6. СОХРАНЯЕМ ОБЯЗАТЕЛЬНО КАК JPG
                name_without_ext = os.path.splitext(img_name)[0]
                output_path = os.path.join(output_folder, name_without_ext + ".jpg")
                cv2.imwrite(output_path, cropped_img)

    except Exception as e:
        print(f"Ошибка при обработке {img_name}: {e}")

print(f"Готово! Чистые этикетки (в формате JPG) лежат в: {output_folder}")

Найдено изображений: 1860. Начинаем обрезку...


100%|██████████| 1860/1860 [00:22<00:00, 81.08it/s]

Готово! Чистые этикетки (в формате JPG) лежат в: /content/new_labels


In [ ]:
!pip install faiss-cpu

  Using cached faiss_cpu-1.14.3-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.8 kB)
Using cached faiss_cpu-1.14.3-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (18.5 MB)


In [ ]:
import os
import json
import numpy as np
import torch
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image
import faiss
from tqdm import tqdm

# =====================================================================
# 1. НАСТРОЙКА ПУТЕЙ И ПАРАМЕТРОВ
# =====================================================================
# Папка с КРОПАМИ (вырезанными этикетками из предыдущего шага)
IMAGE_FOLDER = '/content/new_labels'
# Куда сохранить векторную базу и файл соответствия
FAISS_INDEX_PATH = '/content/wines_base.index'
MAPPING_JSON_PATH = '/content/wines_mapping.json'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")

# =====================================================================
# 2. ПОДГОТОВКА НЕЙРОСЕТИ (МОДЕЛЬ 2)
# =====================================================================
# Загружаем модель с лучшими предобученными весами на ImageNet
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# ХАК: Подменяем последний слой классификатора на Identity (пустышку).
# Теперь модель не будет предсказывать класс, а вернет чистый вектор фичей на 1280 чисел.
model.classifier = torch.nn.Identity()
model = model.to(device)
model.eval() # Переводим в режим инференса (выключения дропаутов и т.д.)

# ТРАНСФОРМАЦИИ: Картинку нужно привести к стандарту, который понимает EfficientNet
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),       # Размер, на котором обучалась сеть
    transforms.ToTensor(),               # Перевод в тензор PyTorch
    transforms.Normalize(                # Стандартная нормализация ImageNet
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# =====================================================================
# 3. ГЕНЕРАЦИЯ ЭМБЕДДИНГОВ
# =====================================================================
embeddings_list = []
mapping_dict = {} # Будет хранить связь: ID вектора -> Имя файла вина

# Получаем список всех картинок в папке кропов
image_files = [f for f in os.listdir(IMAGE_FOLDER) if f.endswith(('.jpg', '.png', '.jpeg'))]
print(f"Найдено {len(image_files)} этикеток для индексации.")

# Отключаем подсчет градиентов, чтобы сэкономить память и ускорить процесс
with torch.no_grad():
    for idx, img_name in enumerate(tqdm(image_files)):
        img_path = os.path.join(IMAGE_FOLDER, img_name)

        try:
            # Загружаем и трансформируем картинку
            img = Image.open(img_path).convert('RGB')
            tensor = preprocess(img).unsqueeze(0).to(device) # Добавляем размерность батча (1, 3, 224, 224)

            # Извлекаем вектор фичей
            features = model(tensor) # Размерность: [1, 1280]

            # Переводим в обычный numpy-массив и нормализуем (для косинусного сходства)
            vector = features.cpu().numpy().flatten()
            vector = vector / np.linalg.norm(vector) # Делаем длину вектора равной 1

            embeddings_list.append(vector)
            mapping_dict[idx] = img_name # Привязываем индекс к имени файла

        except Exception as e:
            print(f"\nОшибка обработки файла {img_name}: {e}")

# Переводим список векторов в массив numpy типа float32 (требование FAISS)
embeddings_array = np.array(embeddings_list).astype('float32')

# =====================================================================
# 4. СОЗДАНИЕ И СОХРАНЕНИЕ БАЗЫ FAISS
# =====================================================================
dimension = 1280 # Размерность вектора EfficientNet-B0

# Используем IndexFlatIP (Inner Product). Так как мы нормализовали векторы,
# скалярное произведение будет работать как косинусное сходство (метрика похожести)
index = faiss.IndexFlatIP(dimension)
index.add(embeddings_array) # Добавляем наши эмбеддинги в базу

# Сохраняем базу на диск
faiss.write_index(index, FAISS_INDEX_PATH)

# Сохраняем json-маппинг (чтобы знать, какой вектор какому вину принадлежит)
with open(MAPPING_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(mapping_dict, f, ensure_ascii=False, indent=4)

print(f"\nУспех! База данных FAISS создана. Всего проиндексировано объектов: {index.ntotal}")

Используем устройство: cuda
Найдено 1221 этикеток для индексации.


100%|██████████| 1221/1221 [00:15<00:00, 79.63it/s]


Успех! База данных FAISS создана. Всего проиндексировано объектов: 1221


In [14]:
import os
import cv2
import json
import numpy as np
import torch
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image
import faiss
from ultralytics import YOLO

# =====================================================================
# 1. НАСТРОЙКА ПУТЕЙ
# =====================================================================
YOLO_MODEL_PATH = '/content/runs/detect/train3/weights/best.pt'  # Твоя обученная YOLO (Модель 1)
FAISS_INDEX_PATH = '/content/wines_base.index'                   # База векторов (из Шага 3)
MAPPING_JSON_PATH = '/content/wines_mapping.json'                 # Словарь соответствия ID -> Имя файла
TEST_IMAGE_PATH = '/content/test3.jpg'               # Любое неразмеченное фото бутылки из магазина

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Работаем на: {device}")

# =====================================================================
# 2. ИНИЦИАЛИЗАЦИЯ И ЗАГРУЗКА ВСЕХ СИСТЕМ
# =====================================================================
print("Загрузка моделей...")
# Загружаем Модель 1 (Детектор)
yolo_model = YOLO(YOLO_MODEL_PATH)

# Загружаем Модель 2 (Эмбеддер)
weights = EfficientNet_B0_Weights.DEFAULT
embedder_model = efficientnet_b0(weights=weights)
embedder_model.classifier = torch.nn.Identity()  # Отрезаем голову классификатора
embedder_model = embedder_model.to(device)
embedder_model.eval()

# Загружаем FAISS индекс и маппинг имен
faiss_index = faiss.read_index(FAISS_INDEX_PATH)
with open(MAPPING_JSON_PATH, 'r', encoding='utf-8') as f:
    wine_mapping = json.load(f)

# Трансформации для EfficientNet
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Все системы загружены и готовы к тесту!\n" + "="*50)

# =====================================================================
# ШАГ 1: ДЕТЕКЦИЯ И ОБРЕЗКА (Модель 1 - YOLO)
# =====================================================================
print("Шаг 1: Поиск этикетки на фото...")
# Считываем оригинальное изображение
original_img = cv2.imread(TEST_IMAGE_PATH)
if original_img is None:
    raise FileNotFoundError(f"Не удалось загрузить тестовое изображение по пути {TEST_IMAGE_PATH}")

# Запускаем YOLO
yolo_results = yolo_model(TEST_IMAGE_PATH, verbose=False)
boxes = yolo_results[0].boxes

# Проверяем, нашла ли YOLO хоть одну этикетку
if len(boxes) == 0:
    print("⚠️ YOLO не нашла этикетку! Используем всё изображение целиком как фолбек (fallback).")
    cropped_bgr = original_img
else:
    # Берем рамку с самым высоким confidence score
    best_box = boxes[0].xyxy[0].tolist()
    x1, y1, x2, y2 = map(int, best_box)

    # Вырезаем этикетку
    cropped_bgr = original_img[y1:y2, x1:x2]
    print(f"🎉 Этикетка успешно найдена и вырезана! Координаты: [{x1}, {y1}, {x2}, {y2}]")

# =====================================================================
# ШАГ 2: СОЗДАНИЕ ЭМБЕДДИНГА (Модель 2 - EfficientNet)
# =====================================================================
print("Шаг 2: Генерация цифрового отпечатка (вектора)...")
# Конвертируем BGR (OpenCV) в RGB (Pillow) для нейросети
cropped_rgb = cv2.cvtColor(cropped_bgr, cv2.COLOR_BGR2RGB)
pil_img = Image.fromarray(cropped_rgb)

# Применяем предобработку и добавляем размерность батча
tensor = preprocess(pil_img).unsqueeze(0).to(device)

# Прогоняем через эмбеддер
with torch.no_grad():
    features = embedder_model(tensor)

# Переводим в нормализованный numpy-вектор
test_vector = features.cpu().numpy().flatten()
test_vector = test_vector / np.linalg.norm(test_vector)  # L2 нормализация
test_vector = np.array([test_vector]).astype('float32')

# =====================================================================
# ШАГ 3: ПОИСК ПО БАЗЕ (FAISS)
# =====================================================================
print("Шаг 3: Поиск совпадений в базе...")
k = 3  # Сколько похожих вариантов выдать
distances, indices = faiss_index.search(test_vector, k)

# Выводим результаты
print("\n" + "="*15 + " РЕЗУЛЬТАТЫ РАСПОЗНАВАНИЯ " + "="*15)
found_any = False
for i in range(k):
    vector_id = indices[0][i]
    score = distances[0][i]  # Для нормализованных векторов это косинусное сходство (от 0 до 1)

    if vector_id != -1:
        found_any = True
        wine_file_name = wine_mapping[str(vector_id)]
        # Косинусное сходство > 0.6 обычно говорит о хорошем совпадении
        match_status = "🔥 ОТЛИЧНОЕ СОВПАДЕНИЕ" if score > 0.7 else "🤔 Возможно, оно"
        print(f"Место {i+1}: {wine_file_name}")
        print(f"   -> Сходство: {score:.4f} ({match_status})")

if not found_any:
    print("❌ В базе данных не найдено похожих вин.")

Работаем на: cuda
Загрузка моделей...
Все системы загружены и готовы к тесту!
Шаг 1: Поиск этикетки на фото...
🎉 Этикетка успешно найдена и вырезана! Координаты: [68, 1102, 558, 1688]
Шаг 2: Генерация цифрового отпечатка (вектора)...
Шаг 3: Поиск совпадений в базе...

=============== РЕЗУЛЬТАТЫ РАСПОЗНАВАНИЯ ===============
Место 1: aratti-kaberne-po-belomu.jpg
   -> Сходство: 0.6317 (🤔 Возможно, оно)
Место 2: solnechnaya-dolina-meganom-beloe-aligote-suhoe-13.jpg
   -> Сходство: 0.6242 (🤔 Возможно, оно)
Место 3: oleg-repin-roze-oleg-repin-sira-rozovoe-suhoe-13.jpg
   -> Сходство: 0.6187 (🤔 Возможно, оно)
